# E1: a new declared nine-atom chemical authority

Owner: BindingMT #78. This is a newly constructed source from the SMILES already
declared in the human 3PTB dossier. It does not restore the legacy snapshots or
the thirteen-atom prepared ligand. Five historical heavy-atom coordinate sets
are received onto this new authority using the already declared axis mapping.
Positive witnesses apply to this new authority and its own variants only.

Run in a fresh disposable process after registering exact code/inputs and
resources. Supply BINDINGMT_E1_ADMISSION, BINDINGMT_E1_PARTIAL_EVIDENCE,
BINDINGMT_E1_PARTIAL_RECEIPT_SHA256 and a new BINDINGMT_E1_AUTHORITY_OUTPUT.


In [ ]:
# Select the small ligand records before importing scientific libraries.
# The complete historical admission is 97 MB decoded and is parsed once only.
from copy import deepcopy
from functools import wraps
import gc
import gzip
import hashlib
import importlib.metadata as metadata
import json
import os
from pathlib import Path
import platform
import sys
from types import SimpleNamespace

admission_path = Path(os.environ['BINDINGMT_E1_ADMISSION'])
partial_evidence = Path(os.environ['BINDINGMT_E1_PARTIAL_EVIDENCE'])
output = Path(os.environ['BINDINGMT_E1_AUTHORITY_OUTPUT'])
output.mkdir(parents=True, exist_ok=False)
packed = admission_path.read_bytes()
assert hashlib.sha256(packed).hexdigest() == '79e9283236ebbb645cd75d7c3341d457a8078361466b7a26458c814d8041cd53'
decoded = gzip.decompress(packed)
admission = json.loads(decoded)
selection = {
    'schema': 'bindingmt.e1_declared_source_selection@1',
    'source_compressed_sha256': hashlib.sha256(packed).hexdigest(),
    'source_decoded_sha256': hashlib.sha256(decoded).hexdigest(),
    'source_paths': ['ligand.observed_snapshot', 'ligand.charged_snapshot',
                     'ligand.template_options'],
    'observed_snapshot': admission['ligand']['observed_snapshot'],
    'charged_snapshot': admission['ligand']['charged_snapshot'],
    'template_options': admission['ligand']['template_options'],
}
(output / 'source_selection.json').write_text(json.dumps(selection, indent=1, allow_nan=False) + '\n')
del admission, decoded
_ = gc.collect()
observed = selection['observed_snapshot']
charged = selection['charged_snapshot']
assert observed['chemical_states']['n_atoms'] == 9
assert charged['chemical_states']['n_atoms'] == 18
assert observed['schema'] == charged['schema'] == 'bindingmt.chemical_template_snapshot@2'
assert selection['template_options']['template_provenance']['source_uri'] == 'smiles:NC(=[NH2+])c1ccccc1'
# Null masks, not placeholder strings, carry the historical unknown stereo.
old_stereo = charged['chemical_states']['states'][0]['atom_attributes']['columns']['stereochemistry']
assert old_stereo['null_mask'] == [True] * 9 + [False] * 9
original_selection = deepcopy(selection)
partial_receipt_bytes = (partial_evidence / 'receipt.json').read_bytes()
assert hashlib.sha256(partial_receipt_bytes).hexdigest() == os.environ['BINDINGMT_E1_PARTIAL_RECEIPT_SHA256']
partial_receipt = json.loads(partial_receipt_bytes)
assert partial_receipt['original_result_compressed_sha256'] == '4b1f51cdeb135fd6785a181f6bb09a0d0f631ec3d9de7c169c88703748924249'
partial_rows = next(item for item in partial_receipt['profiles'] if item['profile'] == 'normal')['poses']
assert len(partial_rows) == 5
assert [row['original_position'] for row in partial_rows] == [1, 2, 3, 4, 5]
for item in partial_rows:
    payload = (partial_evidence / 'normal' / f"pose_{item['original_position']}.pdbqt").read_bytes()
    assert hashlib.sha256(payload).hexdigest() == item['output_sha256']


In [ ]:
import numpy as np
import bindingmt as dmt
import molsysmt as msm
import pyunitwizard as puw
from molsysmt.native import Structures

budgets = {'convert': 80, 'get': 300, 'copy': 30, 'set': 30,
           'extract': 12, 'permute_atoms': 6,
           'get_atom_correspondences': 16, 'get_chemical_readiness': 6}
counts = dict.fromkeys(budgets, 0)
api = SimpleNamespace()

def counted(name, function):
    @wraps(function)
    def call(*args, **kwargs):
        if counts[name] >= budgets[name]:
            raise RuntimeError('Registered direct-call budget exceeded: ' + name)
        counts[name] += 1
        return function(*args, **kwargs)
    return call

for name in ['convert', 'get', 'copy', 'set', 'extract', 'permute_atoms']:
    setattr(api, name, counted(name, getattr(msm, name)))
api.get_atom_correspondences = counted('get_atom_correspondences', msm.chemical_states.get_atom_correspondences)
api.get_chemical_readiness = counted('get_chemical_readiness', msm.physchem.get_chemical_readiness)

def inspection_json(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if puw.is_quantity(value):
        return puw.convert(value, to_form='record').to_dict()
    raise TypeError('Unsupported inspection value: ' + type(value).__name__)

def save(path, value):
    with path.open('x') as stream:
        json.dump(value, stream, default=inspection_json, indent=2, allow_nan=False)
        stream.write('\n')

def refusal(label, operation, exception_type, marker):
    try:
        operation()
    except exception_type as error:
        assert marker.lower() in str(error).lower(), str(error)
        return {'control': label, 'disposition': 'refused',
                'exception_type': type(error).__name__,
                'code': getattr(error, 'code', None), 'message': str(error)}
    raise AssertionError('Expected refusal did not occur: ' + label)

identity = np.column_stack((np.arange(9), np.arange(9)))

def witness(query, reference, candidate):
    report = api.get_atom_correspondences(
        query, reference_molecular_system=reference,
        structure_indices=[0], reference_structure_indices=[0],
        chemical_state='reference', reference_chemical_state='reference',
        candidate_atom_correspondences=candidate, max_matches=1)
    assert report['n_candidate_matches'] == 1
    assert report['search_scope'] == 'supplied_atom_correspondences'
    assert report['search_complete']
    np.testing.assert_array_equal(report['atom_correspondences'], candidate[None])
    return report

# assess/apply_chemical_template pairs are (template index, observed-source index).
# permute_atoms accepts the inverse, (new observed axis -> old template axis).
new_to_old = [3, 4, 5, 6, 7, 8, 1, 2, 0]
old_mapping = np.asarray(selection['template_options']['atom_correspondence'], dtype=int)
np.testing.assert_array_equal(old_mapping[:, 0], np.arange(9))
np.testing.assert_array_equal(old_mapping[new_to_old, 1], np.arange(9))
old_elements = [row['atom_type'] for row in observed['topology']['atoms']]
old_names = [row['atom_name'] for row in observed['topology']['atoms']]
assert old_elements == ['C'] * 7 + ['N'] * 2
old_formal = charged['chemical_states']['states'][0]['atom_attributes']['columns']['formal_charge']
assert not any(old_formal['null_mask'])
old_formal_heavy = old_formal['values'][:9]


In [ ]:
profiles = [
    ('normal', ['nm', 'ps', 'kelvin', 'mole', 'dalton', 'e', 'kJ/mol',
                'kJ/(mol*nm)', 'kJ/(mol*nm**2)', 'radians']),
    ('pm_fs', ['pm', 'fs', 'kelvin', 'mole', 'dalton', 'coulomb', 'kJ/mol',
               'kJ/(mol*pm)', 'kJ/(mol*pm**2)', 'radians']),
]
caller_policy = puw.configure.report()
observations = []
for profile, units in profiles:
    directory = output / profile
    directory.mkdir()
    with puw.context(standard_units=units):
        profile_policy = puw.configure.report()
        legacy_controls = []
        for name in ['observed_snapshot', 'charged_snapshot']:
            legacy_controls.append(refusal(
                name + '_JSON_history_transport',
                lambda name=name: api.convert(
                    msm.ChemicalStatesDict(data=deepcopy(selection[name]['chemical_states'])),
                    to_form='molsysmt.ChemicalStates'),
                ValueError, 'Preparation-history array dtype'))

        # This parses the declared SMILES afresh. It is not a legacy restoration.
        template = api.convert(api.convert('smiles:NC(=[NH2+])c1ccccc1', to_form='rdkit.Mol'),
                               to_form='molsysmt.MolSys')
        assert api.get(template, n_atoms=True) == 9
        ordered_result = api.permute_atoms(template, atom_indices=new_to_old, return_report=True)
        ordered = ordered_result['molecular_system']
        mapping = ordered_result['report']
        np.testing.assert_array_equal(mapping['output_to_source_atom_indices'], new_to_old)
        np.testing.assert_array_equal(mapping['source_to_output_atom_indices'], old_mapping[:, 1])
        np.testing.assert_array_equal(api.get(ordered, element='atom', atom_type=True), old_elements)
        np.testing.assert_array_equal(api.get(ordered, element='atom', formal_charge=True), old_formal_heavy)
        fields = observed['structures']
        structure_values = {'structure_id': fields['structure_id']}
        for field in ['coordinates', 'box', 'time']:
            structure_values[field] = (None if fields[field] is None else
                puw.quantity(fields[field], observed['structure_units'][field]))
        structures = Structures(**structure_values)
        base = api.convert([ordered.topology, ordered.chemical_states, structures],
                           to_form='molsysmt.MolSys')
        api.set(base, structure_chemical_state_index=observed['structure_chemical_state_indices'])
        assert api.get(base, n_atoms=True) == 9 and api.get(base, n_structures=True) == 1
        assert api.get(base, structure_chemical_state_index=True) == [0]
        before_states = api.convert(base.chemical_states, to_form='molsysmt.ChemicalStatesDict').to_dict()
        before_xyz = np.asarray(puw.get_value(api.get(base, element='atom', coordinates=True), to_unit='nm')).copy()
        np.testing.assert_allclose(before_xyz, puw.get_value(structure_values['coordinates'], to_unit='nm'),
                                   rtol=0, atol=1e-12)
        source_labels = {'ids': list(api.get(base, element='atom', atom_id=True)),
                         'names': list(api.get(base, element='atom', name=True)),
                         'elements': list(api.get(base, element='atom', atom_type=True))}
        readiness = api.get_chemical_readiness(base, structure_indices=[0], chemical_state='reference')
        save(directory / 'source_readiness.json', readiness)
        save(directory / 'declared_axis_map.json', mapping)

        native_path = directory / 'declared_authority9.h5msm'
        api.convert(base, to_form=native_path)
        reread = api.convert(native_path, to_form='molsysmt.MolSys')
        roundtrip_witness = witness(reread, base, identity)
        save(directory / 'roundtrip_witness.json', roundtrip_witness)
        np.testing.assert_allclose(puw.get_value(api.get(reread, element='atom', coordinates=True), to_unit='nm'),
                                   before_xyz, rtol=0, atol=1e-12)
        for attribute, expected in [('atom_id', source_labels['ids']),
                                    ('name', source_labels['names']),
                                    ('atom_type', source_labels['elements'])]:
            np.testing.assert_array_equal(api.get(reread, element='atom', **{attribute: True}), expected)
        assert api.get(reread, structure_chemical_state_index=True) == [0]

        entries = []
        for row in partial_rows:
            position = row['original_position']
            payload = (partial_evidence / 'normal' / f'pose_{position}.pdbqt').read_bytes()
            assert hashlib.sha256(payload).hexdigest() == row['output_sha256']
            partial = api.convert('pdbqt_text:' + payload.decode('utf-8'),
                                  to_form='molsysmt.MolSys', discard_torsion_tree=True)
            heavy = api.extract(partial, selection=list(range(9)))
            np.testing.assert_array_equal(api.get(heavy, element='atom', atom_type=True), old_elements)
            np.testing.assert_array_equal(api.get(heavy, element='atom', name=True), old_names)
            coordinates = api.get(heavy, element='atom', coordinates=True)
            assert coordinates.shape == (1, 9, 3)
            variant = api.copy(base)
            api.set(variant, element='atom', coordinates=coordinates)
            received_coordinates = api.get(variant, element='atom', coordinates=True)
            received_nm = np.asarray(puw.get_value(received_coordinates, to_unit='nm'))
            assert received_nm.shape == (1, 9, 3) and np.isfinite(received_nm).all()
            np.testing.assert_allclose(received_nm, puw.get_value(coordinates, to_unit='nm'),
                                       rtol=0, atol=1e-12)
            report = witness(variant, base, identity)
            save(directory / f'pose_{position}_witness.json', report)
            entries.append({
                'original_position': position, 'pose_id': row['pose_id'],
                'coordinate_source_sha256': row['output_sha256'],
                'original_scores': deepcopy(row['original_scores']),
                'new_authority_atom_indices': list(range(9)),
                'historical_prepared_atom_indices': list(range(9)),
                'coordinates': {'value': received_nm.tolist(), 'unit': 'nm'},
                'coordinate_projection_checked': True,
                'witness_file': f'pose_{position}_witness.json',
                'identity_scope': 'New declared authority and its coordinate variant only',
                'historical_prepared_identity_admitted': False,
            })
        permuted_result = api.permute_atoms(base, atom_indices=list(range(8, -1, -1)), return_report=True)
        permuted = permuted_result['molecular_system']
        permutation = np.asarray(permuted_result['report']['output_to_source_atom_indices'])
        permuted_map = np.column_stack((np.arange(9), permutation))
        permutation_witness = witness(permuted, base, permuted_map)
        save(directory / 'nonidentity_map_witness.json', permutation_witness)
        save(directory / 'nonidentity_axis_report.json', permuted_result['report'])

        isotope_variant = api.copy(base)
        api.set(isotope_variant, element='atom', selection=[0], isotope=13)
        assert api.get(isotope_variant, element='atom', isotope=True)[0] == 13
        isotope_refusal = refusal(
            'incompatible_carbon13_supplied_map',
            lambda: api.get_atom_correspondences(
                isotope_variant, reference_molecular_system=base,
                structure_indices=[0], reference_structure_indices=[0],
                chemical_state='reference', reference_chemical_state='reference',
                candidate_atom_correspondences=identity, max_matches=1),
            msm.StructuralInconsistencyError, 'complete-context extension')
        np.testing.assert_equal(api.convert(base.chemical_states,
            to_form='molsysmt.ChemicalStatesDict').to_dict(), before_states)
        np.testing.assert_array_equal(puw.get_value(api.get(base, element='atom', coordinates=True), to_unit='nm'), before_xyz)
        for attribute, expected in [('atom_id', source_labels['ids']), ('name', source_labels['names']),
                                    ('atom_type', source_labels['elements'])]:
            np.testing.assert_array_equal(api.get(base, element='atom', **{attribute: True}), expected)
        assert puw.configure.report() == profile_policy
        observation = {
            'profile': profile, 'authority': 'new_declared_SMILES_source9',
            'source_labels': source_labels, 'source_atoms': 9,
            'source_graph_payload_unchanged': True,
            'source_checked_inventory_and_coordinates_unchanged': True,
            'unit_policy_preserved': True, 'pose_variants': entries,
            'legacy_transport_controls': legacy_controls,
            'isotope_control': isotope_refusal,
            'nonidentity_witness_file': 'nonidentity_map_witness.json',
            'roundtrip_witness_file': 'roundtrip_witness.json',
            'native_source_file': native_path.name,
            'roundtrip_checked_fields': ['coordinates', 'atom_ids', 'atom_names', 'elements',
                                         'structure_state_association', 'chemical_graph_witness'],
        }
        save(directory / 'observations.json', observation)
        observations.append(observation)
    assert puw.configure.report() == caller_policy
    assert selection == original_selection
    assert admission_path.read_bytes() == packed
    assert (partial_evidence / 'receipt.json').read_bytes() == partial_receipt_bytes


In [ ]:
assert len(observations) == 2
assert counts['get_atom_correspondences'] == 16
distinct_coordinates = len({json.dumps(row['coordinates'], sort_keys=True, allow_nan=False)
                            for row in observations[0]['pose_variants']})
assert distinct_coordinates == 5
for first, second in zip(observations[0]['pose_variants'], observations[1]['pose_variants']):
    np.testing.assert_allclose(first['coordinates']['value'], second['coordinates']['value'], rtol=0, atol=1e-12)
assert not any(name == 'vina' or name.startswith('vina.') for name in sys.modules)
loaded_native = []
package_root = Path(msm.__file__).resolve().parent
for name, module in list(sys.modules.items()):
    filename = getattr(module, '__file__', None)
    if name.startswith('molsysmt.') and filename and filename.endswith('.so'):
        path = Path(filename).resolve()
        loaded_native.append({'module': name, 'file': path.name,
                              'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
                              'inside_selected_provider': path.is_relative_to(package_root),
                              'build_source_correspondence': 'unassessed'})
receipt = {
    'schema': 'bindingmt.e1_declared_authority_receiving@1', 'owner': 'uibcdf/bindingmt#78',
    'scope': 'New nine-atom authority from the declared SMILES, not a restored prepared source',
    'source_uri': selection['template_options']['template_provenance']['source_uri'],
    'source_uri_sha256': hashlib.sha256(selection['template_options']['template_provenance']['source_uri'].encode()).hexdigest(),
    'source_construction': 'Public SMILES/RDKit/native conversion; new molecular interpretation and stored-H/stereo declarations',
    'python': platform.python_version(), 'os': platform.system(), 'architecture': platform.machine(),
    'versions': {name: metadata.version(name) for name in ['bindingmt', 'molsysmt', 'argdigest', 'pyunitwizard', 'numpy', 'rdkit', 'h5py']},
    'loaded_provider_native_modules': loaded_native,
    'profiles': observations, 'completed_profiles': len(observations),
    'distinct_historical_coordinate_sets': distinct_coordinates, 'new_declared_states': 1,
    'coordinate_variant_positive_witnesses': sum(len(p['pose_variants']) for p in observations),
    'nonidentity_map_positive_witnesses': len(observations),
    'source_h5_roundtrip_positive_witnesses': len(observations),
    'isotope_negative_controls': len(observations),
    'legacy_transport_refusals': sum(len(p['legacy_transport_controls']) for p in observations),
    'monitored_direct_consumer_calls': counts,
    'monitored_direct_consumer_budgets': budgets,
    'legacy_snapshots_restored': False, 'historical_prepared13_identity_admitted': False,
    'partial_charges_types_or_tree_received': False, 'score_comparison_admitted': False,
    'e1_accepted': False,
    'excluded_operations': {'direct_calls_in_reviewed_notebook': 0,
        'groups': ['search', 'scoring', 'optimization', 'H_generation', 'partial_charge_assignment',
                   'AutoDock_type_assignment', 'interaction_detectors', 'viewer'],
        'basis': 'Inspected route; excluded APIs are not counter-instrumented'},
    'claim_limit': 'Identity of one new declared graph under supplied candidate maps; coordinate reuse and unit repetitions are not independent complexes or historical-state authentication.'
}
save(output / 'receipt.json', receipt)
print(json.dumps({'positive_witnesses': 14, 'legacy_transport_refusals': 4,
                  'isotope_refusals': 2, 'historical_prepared13_identity': 'unassessed',
                  'energy_calls': 0}))


The legacy JSON snapshots and their unknown fields remain unchanged. The new
source is a separate scientific producer with its own H representation, axis
map, readiness report, H5MSM files and positive witnesses. These witnesses do
not authenticate the original prepared thirteen-atom chemistry, restore partial
charges/types/history or justify energy comparison/ranking. No JSON molecular
codec, guessed stereo labels or hydrogen normalization is implemented here.
